# Análisis y Preparación de Datos: Ejemplo Aplicado de Manchas Solares

Este notebook es un ejemplo aplicado de **Análisis y Preparación de Datos** en el contexto de la Ingeniería Mecatrónica. Se realiza una limpieza exhaustiva, análisis comparativo, reducción de dimensionalidad (PCA) y detección de outliers sobre un dataset de clasificación McIntosh de manchas solares.

**Etapas clave:**
1. Carga y Limpieza inicial de nulos.
2. Normalización de metadatos (parámetros físicos).
3. Análisis de componentes principales (PCA).
4. Detección y retiro de outliers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from mpl_toolkits.mplot3d import Axes3D

# Configuración de estética
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Carga y Limpieza Inicial

Se cargan los datos y se realiza el tratamiento de nulos para asegurar la integridad estadística.

In [ ]:
# Carga del CSV
df_raw = pd.read_csv('sunspot_data.csv')

# Eliminación de filas y columnas vacías o con NaNs
print(f"Forma original: {df_raw.shape}")
df = df_raw.dropna(axis=0, how='any').copy()
df = df.dropna(axis=1, how='all')
print(f"Forma tras limpieza inicial (NaNs): {df.shape}")

# Ingeniería de características: Separación de McIntosh
df['mc_Z'] = df['mcintosh_full'].str[0]
df['mc_P'] = df['mcintosh_full'].str[1]
df['mc_C'] = df['mcintosh_full'].str[2]

## 2. Normalización de Metadatos (Parámetros Físicos)

La normalización es crucial para que variables con diferentes unidades (ej. área en pixeles vs elongación) tengan el mismo peso en el análisis.

In [ ]:
# Selección de parámetros físicos (metadatos)
features = ['area', 'dist_angular', 'z_length_deg', 'num_spots_det', 'area_total', 
            'fill_ratio', 'elongation', 'pen_diam_ns_deg', 'pen_symmetry', 'pen_irregularity']

X = df[features]

# Estandarización (Z-score): Media 0, Desviación Estándar 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertimos a DataFrame para visualización
df_norm = pd.DataFrame(X_scaled, columns=features)
print("Primeras filas de los metadatos normalizados:")
display(df_norm.head())

## 3. Visualización y Estadísticos

Comparamos las distribuciones y visualizamos las medias agrupadas.

In [ ]:
def plot_mcintosh_histograms(data, title_suffix=""):
    fig, axes = plt.subplots(1, 4, figsize=(24, 5))
    data['mcintosh_full'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue')
    axes[0].set_title(f'McIntosh Full {title_suffix}')
    
    for i, char in enumerate(['mc_Z', 'mc_P', 'mc_C']):
        data[char].value_counts().sort_index().plot(kind='bar', ax=axes[i+1], color='salmon')
        axes[i+1].set_title(f'Carácter {char[-1]} {title_suffix}')
    plt.tight_layout()
    plt.show()

plot_mcintosh_histograms(df, "(Antes)")

print("Resumen Visual de Medias por Clase (Datos Originales):")
stats_summary = df.groupby('mcintosh_full')[features].mean()
try:
    display(stats_summary.style.background_gradient(cmap='YlGnBu').format("{:.2f}"))
except Exception:
    display(stats_summary)

## 4. PCA y Detección de Outliers

Reducimos la dimensionalidad para visualizar anomalías en un espacio 3D.

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print(f"Varianza total explicada por 3 componentes: {np.sum(pca.explained_variance_ratio_):.4f}")

# Detección con Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
df['outlier'] = iso.fit_predict(X_scaled)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
normal = X_pca[df['outlier'] == 1]
anomaly = X_pca[df['outlier'] == -1]

ax.scatter(normal[:, 0], normal[:, 1], normal[:, 2], c='blue', alpha=0.3, label='Normal')
ax.scatter(anomaly[:, 0], anomaly[:, 1], anomaly[:, 2], c='red', marker='x', s=50, label='Outlier')
ax.set_title('PCA con Identificación de Outliers')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')
ax.legend()
plt.show()

## 5. Retiro de Outliers y Exportación

Generamos la base de datos limpia final para futuros análisis o entrenamiento de modelos.

In [ ]:
df_clean = df[df['outlier'] == 1].copy()
print(f"Dataset final limpio: {df_clean.shape} registros.")

plot_mcintosh_histograms(df_clean, "(Después)")

df_clean.to_csv('sunspot_data_clean.csv', index=False)
print("Archivo 'sunspot_data_clean.csv' exportado.")